# NoCap-Test: Smart Filtering on Google Colab

Trains GPT-2 124M on FineWeb with Smart Filtering (see `IDEA.md`) and, optionally, the unmodified baseline
on the **same GPU**.

**Runtime → Change runtime type → GPU (T4).**

- **`full` + `RUN_BASELINE = False`** (default): the real benchmark, Smart Filtering only, compared with the
  official baseline (3.3821 at step 4768). ~45 h on a free T4 → several sessions; just **Run all** again after a
  disconnect, it resumes from the last checkpoint on Google Drive automatically.
- **`quick` + `RUN_BASELINE = True`**: a short side-by-side comparison (~2.7 h per run on a T4).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 1. Setup

In [ ]:
import os
USE_DRIVE = True  #@param {type:"boolean"}
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/nocap'   # logs + checkpoints survive a disconnect
else:
    OUT = '/content/nocap-out'
os.makedirs(OUT, exist_ok=True)

SRC = '/content/BottleCapAI_NoCap-Test'
if not os.path.exists(SRC):
    !git clone https://github.com/vbtronic/BottleCapAI_NoCap-Test.git {SRC}
%cd {SRC}
!git pull -q
!pip install -q huggingface_hub

### 2. Configuration

| preset | tokens/step | steps | T4 | L4 | A100 |
|---|---|---|---|---|---|
| `quick` | 131k | 1200 | ~2.7 h | ~1 h | ~25 min |
| `full` | 524k | 4768, stops at val 3.3821 | ~45 h (several sessions) | ~16 h | ~4 h |

Checkpoints go to Google Drive every `SAVE_EVERY` steps (full: 50 ≈ 30 min on a T4), so a disconnect
loses at most that much. Keep `USE_DRIVE` on.

In [ ]:
PRESET = "full"        #@param ["quick", "full"]
RUN_BASELINE = False   #@param {type:"boolean"}
POOL = 1.25            #@param {type:"number"}
SCORE_LEN = 64         #@param {type:"integer"}
BATCH_SIZE = 16        #@param {type:"integer"}
SAVE_EVERY = 50 if PRESET == "full" else 200

import sys, glob, importlib
sys.path.insert(0, SRC)
import experiments as ex
importlib.reload(ex)
RUN_DIR = f"{OUT}/{PRESET}"
os.makedirs(RUN_DIR, exist_ok=True)

def latest_checkpoint(name):
    found = sorted(glob.glob(f"{RUN_DIR}/{name}/ckpt/*/latest.pt"), key=os.path.getmtime)
    return found[-1] if found else ""

def cmd(name, pool=None, resume="auto"):
    if resume == "auto":
        resume = latest_checkpoint(name)
    print(f"{name}: " + (f"resuming from {resume}" if resume else "starting from scratch"))
    c = ex.train_cmd(PRESET, f"{RUN_DIR}/{name}", f"{RUN_DIR}/{name}/ckpt", batch_size=BATCH_SIZE,
                     pool=pool, score_len=SCORE_LEN if pool else None, extra=f"--save_every {SAVE_EVERY}")
    if resume:
        c += f" --resume {resume}"
    return c + f" 2>&1 | tee -a {RUN_DIR}/{name}.txt"

print("shards to download:", ex.shards_needed(PRESET, POOL))

### 3. Data
Each shard is 100M tokens (~200 MB). Smart Filtering reads `POOL`× more data than it trains on.

In [ ]:
!python data/cached_fineweb10B.py {ex.shards_needed(PRESET, POOL)}
!du -sh data/fineweb10B

### 4. Baseline (only if `RUN_BASELINE`)
`RESUME = "auto"` continues from the newest checkpoint on Drive; `""` starts over; or paste a `latest.pt` path.

In [ ]:
RESUME = "auto"  #@param {type:"string"}
if RUN_BASELINE:
    !{cmd("baseline", resume=RESUME)}
else:
    print("skipped (RUN_BASELINE = False)")

### 5. Smart Filtering
After a disconnect: reconnect, **Run all** — it resumes automatically.

In [ ]:
RESUME = "auto"  #@param {type:"string"}
!{cmd("smartfilter", pool=POOL, resume=RESUME)}

### 6. Results
With the baseline: `time ratio` = steps ratio × step cost. Without it (full): steps to 3.3821 ÷ 4768 × step cost.
Below 1.0 Smart Filtering is faster.

In [ ]:
if RUN_BASELINE:
    ex.evaluate([f"{RUN_DIR}/baseline.txt", f"{RUN_DIR}/smartfilter.txt"], plot=f"{RUN_DIR}/compare.png")
    from IPython.display import Image, display
    display(Image(f"{RUN_DIR}/compare.png"))
else:
    ex.evaluate_full([f"{RUN_DIR}/smartfilter.txt"])